In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
emp = pd.read_csv("data/Employee.csv")
perf = pd.read_csv("data/PerformanceRating.csv")
df = emp.merge(perf, on="EmployeeID", how="left")
df = df.drop([
    "HireDate",
    "ReviewDate"
], axis=1)
df = df.dropna(subset=["Attrition"])

In [ ]:
le = LabelEncoder()
for col in df.columns:
    if df[col].dtype == "object" and col not in ["EmployeeID", "FirstName", "LastName"]:
        df[col] = le.fit_transform(df[col])

df = df.dropna()
X = df.drop(columns=["Attrition", "EmployeeID", "FirstName", "LastName"])
y = df["Attrition"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Logistic Regression Model
log_model = LogisticRegression(max_iter=300)
log_model.fit(X_train, y_train)
y_pred_log = log_model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred_log))
print(classification_report(y_test, y_pred_log))

In [ ]:
# Random Forest Model
rf = RandomForestClassifier(n_estimators=300, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

In [ ]:
# Feature Importance
importances = pd.Series(rf.feature_importances_, index=X.columns)
print(importances.sort_values(ascending=False).head(15))

In [ ]:
test_probs = rf.predict_proba(X_test)[:, 1]
export = X_test.copy()

export["Attrition"] = y_test.values
export["Attrition_Probability"] = test_probs
export["Attrition_Prediction"] = rf.predict(X_test)
export["EmployeeID"] = df.loc[X_test.index, "EmployeeID"].values

emp_names = emp[["EmployeeID", "FirstName", "LastName"]]
export["EmployeeID"] = export["EmployeeID"].astype(str)
emp_names["EmployeeID"] = emp_names["EmployeeID"].astype(str)

export = export.merge(emp_names, on="EmployeeID", how="left")
cols = [
    "EmployeeID", "FirstName", "LastName",
    "Attrition", "Attrition_Prediction", "Attrition_Probability"
] + [c for c in export.columns if c not in [
    "EmployeeID", "FirstName", "LastName",
    "Attrition", "Attrition_Prediction", "Attrition_Probability"
]]
export = export[cols]
export.to_csv("employee_attrition_predictions.csv", index=False)
print("Saved: employee_attrition_predictions.csv")


In [ ]:
feature_importance = pd.DataFrame({
    "Feature": importances.index,
    "Importance": importances.values
})
feature_importance.to_csv("feature_importance.csv", index=False)